In [32]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [33]:
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))

Current working directory: /home/sagemaker-user
Files here: ['.bashrc', '.sagemaker_sql_editor_api_cache', '.local', '.ipython', '.aws', '.npm', '.jupyter', '.ipynb_checkpoints', '.cache', '.config', 'netflix-data.zip', 'netflix_data', 'dataprep', 'training', 'data_exploration_netflix.ipynb', 'data_prep_netflix.ipynb', 'Untitled-Copy1.ipynb', 'factorization_model.ipynb', 'train.csv', 'validation.csv', 'model_evaluations.ipynb', 'user-default-efs', '.virtual_documents']


In [3]:
for root, dirs, files in os.walk("."):
    if "test.csv" in files:
        print("Found at:", os.path.join(root, "test.csv"))

Found at: ./training/test.csv


In [4]:
train_df = pd.read_csv("train.csv", header=None, names=["rating", "user_id", "movie_id"])
validation_df = pd.read_csv("validation.csv", header=None, names=["rating", "user_id", "movie_id"])

test_full = pd.read_csv("training/test.csv")
test_df = test_full[["rating", "user_id", "movie_id"]].copy()

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

test_df.head()

Train: (280000, 3)
Validation: (60000, 3)
Test: (40000, 3)


,rating,user_id,movie_id
0,4,1780095,13384
1,5,287034,13384
2,3,821222,13384
3,5,2194798,13384
4,5,1379931,13384


In [5]:
def evaluate_predictions(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return pd.DataFrame({
        "Model": [model_name],
        "RMSE": [rmse],
        "MAE": [mae]
    })

In [6]:
global_mean = train_df["rating"].mean()

user_bias = train_df.groupby("user_id")["rating"].mean() - global_mean
movie_bias = train_df.groupby("movie_id")["rating"].mean() - global_mean

print("Global mean:", global_mean)

Global mean: 3.548460714285714


In [7]:
test_bias = test_df.copy()

test_bias["user_bias"] = test_bias["user_id"].map(user_bias).fillna(0)
test_bias["movie_bias"] = test_bias["movie_id"].map(movie_bias).fillna(0)

test_bias["prediction"] = global_mean + test_bias["user_bias"] + test_bias["movie_bias"]
test_bias["prediction"] = test_bias["prediction"].clip(1, 5)

bias_results = evaluate_predictions(
    test_bias["rating"],
    test_bias["prediction"],
    "User-Movie Bias Model"
)

bias_results

,Model,RMSE,MAE
0,User-Movie Bias Model,0.859569,0.622555


In [8]:
user_item_train = train_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print(user_item_train.shape)
user_item_train.head()

(142032, 101)


movie_id,1,2,3,4,5,6,7,8,9,10,...,13380,13381,13382,13383,13384,13385,13386,13387,13388,13389
user_id,,,,,,,,,,,,,,,,,,,,,
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,...,NaN,NaN,NaN,5.0,5.0,NaN,NaN,NaN,4.0,NaN
59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN
97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
user_item_filled = user_item_train.apply(
    lambda row: row.fillna(row.mean()),
    axis=1
)

# if a user row is entirely NaN, fall back to the global mean
global_mean = train_df["rating"].mean()
user_item_filled = user_item_filled.fillna(global_mean)

print(user_item_filled.shape)
user_item_filled.head()

(142032, 101)


movie_id,1,2,3,4,5,6,7,8,9,10,...,13380,13381,13382,13383,13384,13385,13386,13387,13388,13389
user_id,,,,,,,,,,,,,,,,,,,,,
6,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.0,3.000000,3.000000,...,3.000000,3.000000,3.000000,3.0,3.0,3.000000,3.000000,3.000000,3.0,3.000000
7,4.444444,4.444444,4.444444,4.444444,4.444444,4.444444,4.444444,5.0,4.444444,4.444444,...,4.444444,4.444444,4.444444,5.0,5.0,4.444444,4.444444,4.444444,4.0,4.444444
59,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.0,3.000000,3.000000,...,3.000000,3.000000,3.000000,4.0,3.0,3.000000,3.000000,3.000000,3.0,3.000000
79,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,3.5,3.500000,3.500000,...,3.500000,3.500000,3.500000,3.5,4.0,3.500000,3.500000,3.500000,3.5,3.500000
97,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.0,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.0,4.0,4.000000,4.000000,4.000000,4.0,4.000000


In [10]:
from sklearn.decomposition import NMF

nmf_model = NMF(
    n_components=20,
    init="random",
    random_state=42,
    max_iter=200
)

W = nmf_model.fit_transform(user_item_filled)
H = nmf_model.components_

print("W shape:", W.shape)
print("H shape:", H.shape)

W shape: (142032, 20)
H shape: (20, 101)


/opt/conda/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [11]:
reconstructed = np.dot(W, H)

nmf_predictions = pd.DataFrame(
    reconstructed,
    index=user_item_filled.index,
    columns=user_item_filled.columns
)

nmf_predictions.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,13380,13381,13382,13383,13384,13385,13386,13387,13388,13389
user_id,,,,,,,,,,,,,,,,,,,,,
6,3.026188,3.012341,2.979998,2.929607,3.019889,2.977434,2.908516,3.050206,3.003241,2.974647,...,3.129255,2.991186,2.997215,2.891109,2.980778,3.003739,2.977029,3.050363,3.261126,3.010843
7,4.383866,4.440828,4.419959,4.453269,4.419046,4.538046,4.438838,5.041054,4.437125,4.487191,...,4.447850,4.455695,4.496840,4.593579,5.024285,4.485353,4.415690,4.424874,4.264294,4.574029
59,3.005654,3.003770,3.008409,3.023261,3.004003,2.967032,3.023381,3.078353,3.009807,2.979929,...,2.990321,2.980676,2.993871,3.194337,2.982162,3.008505,3.017411,2.980158,3.018746,3.235709
79,3.502787,3.501010,3.503067,3.502618,3.498868,3.492042,3.500392,3.501148,3.501400,3.495637,...,3.505016,3.486857,3.492286,3.497627,3.995180,3.502455,3.501283,3.494939,3.504493,3.482798
97,3.998483,4.000214,4.003191,4.001945,3.999734,4.003668,4.002845,3.990243,3.998822,4.002392,...,4.000099,4.000225,3.997200,4.023399,4.001665,3.999630,3.998556,3.998664,4.018749,3.977478


In [13]:
test_nmf = test_df.copy()

global_mean = train_df["rating"].mean()

def predict_nmf(user_id, movie_id):
    if user_id in nmf_predictions.index and movie_id in nmf_predictions.columns:
        return nmf_predictions.loc[user_id, movie_id]
    return global_mean

test_nmf["prediction"] = test_nmf.apply(
    lambda row: predict_nmf(row["user_id"], row["movie_id"]),
    axis=1
)

test_nmf["prediction"] = test_nmf["prediction"].clip(1, 5)

nmf_results = evaluate_predictions(
    test_nmf["rating"],
    test_nmf["prediction"],
    "NMF / Matrix Factorization"
)

nmf_results

,Model,RMSE,MAE
0,NMF / Matrix Factorization,0.731452,0.382565


In [14]:
evaluation_results = pd.concat([bias_results, nmf_results], ignore_index=True)
evaluation_results

,Model,RMSE,MAE
0,User-Movie Bias Model,0.859569,0.622555
1,NMF / Matrix Factorization,0.731452,0.382565


In [15]:
best_baseline = evaluation_results.sort_values("RMSE").iloc[0]["Model"]

print("Baseline comparison complete.")
print("Best baseline by RMSE:", best_baseline)

Baseline comparison complete.
Best baseline by RMSE: NMF / Matrix Factorization


In [16]:
import sagemaker
from sagemaker import image_uris, model_uris, script_uris
from sagemaker.estimator import Estimator
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import CSVSerializer
from sagemaker.predictor import Predictor

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

print("Region:", region)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Role: arn:aws:iam::665606011923:role/LabRole


In [17]:
bucket = "aaron-netflix-ads508-665606011923-us-east-1-an"

fm_model_artifact = (
    f"s3://{bucket}/models/"
    "factorization-machines-2026-04-06-19-49-14-609/output/model.tar.gz"
)

print(fm_model_artifact)

s3://aaron-netflix-ads508-665606011923-us-east-1-an/models/factorization-machines-2026-04-06-19-49-14-609/output/model.tar.gz


In [18]:
fm_inference_image = sagemaker.image_uris.retrieve(
    framework="factorization-machines",
    region=region
)

print(fm_inference_image)

382416733822.dkr.ecr.us-east-1.amazonaws.com/factorization-machines:1


In [19]:
from sagemaker.model import Model

fm_model = Model(
    image_uri=fm_inference_image,
    model_data=fm_model_artifact,
    role=role,
    sagemaker_session=sess
)

In [20]:
fm_predictor = fm_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer()
)

print(type(fm_predictor))
print(fm_predictor)

----------!<class 'NoneType'>
None


In [21]:
#finding endpoint name

sm_client = sess.sagemaker_client

response = sm_client.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending"
)

for ep in response["Endpoints"][:10]:
    print(ep["EndpointName"], ep["EndpointStatus"])

factorization-machines-2026-04-06-23-30-38-827 InService


In [22]:
X_test = test_df[["user_id", "movie_id"]].copy()
y_test = test_df["rating"].copy()

print(X_test.shape)
X_test.head()

(40000, 2)


,user_id,movie_id
0,1780095,13384
1,287034,13384
2,821222,13384
3,2194798,13384
4,1379931,13384


In [23]:
import io
import numpy as np
import sagemaker.amazon.common as smac

X_test_np = X_test.astype("float32").to_numpy()

small_batch_np = X_test_np[:100]

buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, small_batch_np)
payload = buf.getvalue()

print(type(payload), len(payload))

<class 'bytes'> 3200


In [24]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer

endpoint_name = "factorization-machines-2026-04-06-23-30-38-827"

fm_predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
    serializer=IdentitySerializer(content_type="application/x-recordio-protobuf"),
    deserializer=JSONDeserializer()
)

print(type(fm_predictor))
print(fm_predictor.endpoint_name)

<class 'sagemaker.base_predictor.Predictor'>
factorization-machines-2026-04-06-23-30-38-827


In [25]:
small_result = fm_predictor.predict(payload)
small_result

{'predictions': [{'score': -4248993280.0},
  {'score': -632127104.0},
  {'score': -1925873280.0},
  {'score': -5253529088.0},
  {'score': -3280109056.0},
  {'score': -5379358208.0},
  {'score': -4938956288.0},
  {'score': -2314370560.0},
  {'score': -5467438592.0},
  {'score': -2052750976.0},
  {'score': -4892818944.0},
  {'score': -3384966656.0},
  {'score': -761429632.0},
  {'score': -5312249344.0},
  {'score': -5136088576.0},
  {'score': -1402633856.0},
  {'score': -3911351808.0},
  {'score': -5765234176.0},
  {'score': -5345803776.0},
  {'score': 2562716928.0},
  {'score': 5960102912.0},
  {'score': 5465175040.0},
  {'score': 1101264128.0},
  {'score': 5230294016.0},
  {'score': 490665152.0},
  {'score': 6283064320.0},
  {'score': 6299841536.0},
  {'score': 1589638400.0},
  {'score': 1799877888.0},
  {'score': 5045744640.0},
  {'score': 1174140160.0},
  {'score': 4022334720.0},
  {'score': 2186802432.0},
  {'score': 6140457984.0},
  {'score': 4244632832.0},
  {'score': 4297061376.0

In [26]:
small_preds = [p["score"] for p in small_result["predictions"]]
print(len(small_preds))
print(small_preds[:5])

100
[-4248993280.0, -632127104.0, -1925873280.0, -5253529088.0, -3280109056.0]


In [27]:
batch_size = 500
fm_preds = []

for start in range(0, len(X_test_np), batch_size):
    end = start + batch_size
    batch_np = X_test_np[start:end]

    buf = io.BytesIO()
    smac.write_numpy_to_dense_tensor(buf, batch_np)
    batch_payload = buf.getvalue()

    batch_result = fm_predictor.predict(batch_payload)
    batch_scores = [p["score"] for p in batch_result["predictions"]]
    fm_preds.extend(batch_scores)

print("Total predictions:", len(fm_preds))
print("First 5 predictions:", fm_preds[:5])

Total predictions: 40000
First 5 predictions: [-4248993280.0, -632127104.0, -1925873280.0, -5253529088.0, -3280109056.0]


In [28]:
test_fm = test_df.copy()
test_fm["prediction"] = np.clip(fm_preds, 1, 5)

fm_results = evaluate_predictions(
    test_fm["rating"],
    test_fm["prediction"],
    "Factorization Machines"
)

fm_results

,Model,RMSE,MAE
0,Factorization Machines,2.49485,2.173075


In [29]:
final_results = pd.concat([evaluation_results, fm_results], ignore_index=True)
final_results

,Model,RMSE,MAE
0,User-Movie Bias Model,0.859569,0.622555
1,NMF / Matrix Factorization,0.731452,0.382565
2,Factorization Machines,2.494850,2.173075


In [30]:
best_model = final_results.sort_values("RMSE").iloc[0]["Model"]

print("Final model evaluation comparison complete.")
print("Best model by RMSE:", best_model)
print("The NMF / Matrix Factorization model performed best overall on test set.")
print("The Factorization Machines model produced the weakest results.")

Final model evaluation comparison complete.
Best model by RMSE: NMF / Matrix Factorization
The NMF / Matrix Factorization model performed best overall on test set.
The Factorization Machines model produced the weakest results.


In [31]:
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"Could not delete endpoint {endpoint_name}: {e}")

Deleted endpoint: factorization-machines-2026-04-06-23-30-38-827
